In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

## Loading data

In [4]:
data = pd.read_csv("data.csv")
data.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


## Data Exploration

In [292]:
data.shape

(569, 33)

In [293]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [309]:
all(data.duplicated())

False

In [294]:
data.drop(columns=['id', 'Unnamed: 32'], inplace=True)

## Encoding Categorical Variable

In [295]:
X = data.drop(columns=['diagnosis'],axis=1)
y = data['diagnosis']
y.value_counts()

diagnosis
B    357
M    212
Name: count, dtype: int64

In [296]:
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [297]:
data['diagnosis'] = data['diagnosis'].astype('category')

## Feature Selection

In [299]:
# Calculate mutual information (information gain) for each feature
info_gain = mutual_info_classif(X, y, random_state=42)

# Create a DataFrame for easy viewing
feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'Information Gain': info_gain
})

# Sort features by information gain in descending order
feature_scores = feature_scores.sort_values(by='Information Gain', ascending=False)
print(feature_scores)

# Select top N features based on information gain
N = 15
selected_features = feature_scores['Feature'].head(N).tolist()
print("Top features based on Information Gain:", selected_features)


                    Feature  Information Gain
22          perimeter_worst          0.471842
23               area_worst          0.464313
20             radius_worst          0.451230
7       concave points_mean          0.438806
27     concave points_worst          0.436255
2            perimeter_mean          0.402361
6            concavity_mean          0.375447
0               radius_mean          0.362276
3                 area_mean          0.360023
13                  area_se          0.340759
26          concavity_worst          0.315259
12             perimeter_se          0.275614
10                radius_se          0.249301
25        compactness_worst          0.225211
5          compactness_mean          0.213439
17        concave points_se          0.125415
21            texture_worst          0.120331
16             concavity_se          0.117440
1              texture_mean          0.096540
24         smoothness_worst          0.095697
28           symmetry_worst       

In [300]:
# X = X[selected_features]

## Performing train test split

In [302]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Scailing X

In [303]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Decision Tree 

In [304]:
dt = DecisionTreeClassifier(random_state=42)

In [305]:
dt.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [306]:
recall = cross_val_score(dt, X_train, y_train, cv=5, scoring='recall')
print('Recall', np.mean(recall), recall)
precision = cross_val_score(dt, X_train, y_train, cv=5, scoring='precision')
print('Precision', np.mean(precision), precision)
f1 = cross_val_score(dt, X_train, y_train, cv=5, scoring='f1')
print('F1', np.mean(f1), f1)

Recall 0.8996434937611408 [0.93939394 0.79411765 1.         0.91176471 0.85294118]
Precision 0.883123123123123 [0.86111111 0.9        0.85       0.83783784 0.96666667]
F1 0.8881418160352637 [0.89855072 0.84375    0.91891892 0.87323944 0.90625   ]


## Random Forest

In [307]:
rf = RandomForestClassifier()

In [308]:
recall = cross_val_score(rf, X_train, y_train, cv=5, scoring='recall')
print('Recall', np.mean(recall), recall)
precision = cross_val_score(rf, X_train, y_train, cv=5, scoring='precision')
print('Precision', np.mean(precision), precision)
f1 = cross_val_score(rf, X_train, y_train, cv=5, scoring='f1')
print('F1', np.mean(f1), f1)

Recall 0.9292335115864528 [0.96969697 0.85294118 1.         0.94117647 0.88235294]
Precision 0.9586928104575165 [1.         0.96666667 0.94444444 0.94117647 0.94117647]
F1 0.9453094699418229 [0.98461538 0.92307692 0.97142857 0.94117647 0.90625   ]
